# House Prices - Model Industrialization

Refactored notebook with:
- Train/validation split **before** preprocessing (no data leakage)
- `StandardScaler` and `OneHotEncoder` from sklearn (no `fit_transform`, no `ColumnTransformer`/`Pipeline`)
- All preprocessing artifacts and the trained model persisted to `../models/` with `joblib`
- A `Model inference` section that loads everything back from disk and predicts on `test.csv`

## 1. Imports

In [ ]:
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_log_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 2. Constants and paths

In [2]:
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(exist_ok=True)

CONTINUOUS_FEATURES = ["GrLivArea", "TotalBsmtSF"]
CATEGORICAL_FEATURES = ["Neighborhood", "HouseStyle"]
FEATURE_COLUMNS = CONTINUOUS_FEATURES + CATEGORICAL_FEATURES
TARGET_COLUMN = "SalePrice"

TRAIN_PATH = "../data/train.csv"
TEST_PATH = "../data/test.csv"

## 3. RMSLE Function

In [3]:
def compute_rmsle(y_true: np.ndarray, y_pred: np.ndarray, precision: int = 2) -> float:
    """Root Mean Squared Logarithmic Error. Predictions are clipped to >= 1."""
    y_pred = np.maximum(y_pred, 1)
    rmsle = np.sqrt(mean_squared_log_error(y_true, y_pred))
    return round(rmsle, precision)

## 4. Model building

### 4.1 Model training

**Load and split immediately, before any preprocessing (avoids data leakage).**

In [4]:
train_df = pd.read_csv(TRAIN_PATH)
print("Loaded train.csv with shape:", train_df.shape)

X = train_df[FEATURE_COLUMNS].copy()
y = train_df[TARGET_COLUMN].copy()

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_valid shape:", X_valid.shape)

Loaded train.csv with shape: (1460, 81)
X_train shape: (1168, 4)
X_valid shape: (292, 4)


**Missing-value imputation: compute medians/modes from `X_train` only, persist them, then apply to both sets.**

In [5]:
continuous_medians = X_train[CONTINUOUS_FEATURES].median().to_dict()
categorical_modes = {col: X_train[col].mode()[0] for col in CATEGORICAL_FEATURES}

joblib.dump(continuous_medians, MODELS_DIR / "continuous_medians.joblib")
joblib.dump(categorical_modes, MODELS_DIR / "categorical_modes.joblib")

print("Medians:", continuous_medians)
print("Modes:", categorical_modes)

Medians: {'GrLivArea': 1473.0, 'TotalBsmtSF': 997.5}
Modes: {'Neighborhood': 'NAmes', 'HouseStyle': '1Story'}


In [6]:
X_train_filled = X_train.copy()
X_valid_filled = X_valid.copy()

for col in CONTINUOUS_FEATURES:
    X_train_filled[col] = X_train_filled[col].fillna(continuous_medians[col])
    X_valid_filled[col] = X_valid_filled[col].fillna(continuous_medians[col])

for col in CATEGORICAL_FEATURES:
    X_train_filled[col] = X_train_filled[col].fillna(categorical_modes[col])
    X_valid_filled[col] = X_valid_filled[col].fillna(categorical_modes[col])

X_train_filled.isnull().sum()

GrLivArea       0
TotalBsmtSF     0
Neighborhood    0
HouseStyle      0
dtype: int64

**Continuous scaling with `StandardScaler` — fit on train, transform on train and valid separately.**

In [7]:
scaler = StandardScaler()
scaler.fit(X_train_filled[CONTINUOUS_FEATURES])

X_train_continuous = pd.DataFrame(
    scaler.transform(X_train_filled[CONTINUOUS_FEATURES]),
    columns=CONTINUOUS_FEATURES,
    index=X_train_filled.index,
)
X_valid_continuous = pd.DataFrame(
    scaler.transform(X_valid_filled[CONTINUOUS_FEATURES]),
    columns=CONTINUOUS_FEATURES,
    index=X_valid_filled.index,
)

joblib.dump(scaler, MODELS_DIR / "scaler.joblib")
X_train_continuous.head()

,GrLivArea,TotalBsmtSF
254,-0.407093,0.572612
1066,0.083170,-0.596547
638,-1.395250,-0.603357
799,0.458975,-0.750921
380,0.312087,-0.081209


**Categorical encoding with `OneHotEncoder` — fit on train, transform on train and valid separately.**

In [8]:
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoder.fit(X_train_filled[CATEGORICAL_FEATURES])

encoded_columns = encoder.get_feature_names_out(CATEGORICAL_FEATURES)

X_train_categorical = pd.DataFrame(
    encoder.transform(X_train_filled[CATEGORICAL_FEATURES]),
    columns=encoded_columns,
    index=X_train_filled.index,
)
X_valid_categorical = pd.DataFrame(
    encoder.transform(X_valid_filled[CATEGORICAL_FEATURES]),
    columns=encoded_columns,
    index=X_valid_filled.index,
)

joblib.dump(encoder, MODELS_DIR / "encoder.joblib")
print("Encoded train shape:", X_train_categorical.shape)
X_train_categorical.head()

Encoded train shape: (1168, 33)


,Neighborhood_Blmngtn,Neighborhood_Blueste,Neighborhood_BrDale,Neighborhood_BrkSide,Neighborhood_ClearCr,Neighborhood_CollgCr,Neighborhood_Crawfor,Neighborhood_Edwards,Neighborhood_Gilbert,Neighborhood_IDOTRR,...,Neighborhood_Timber,Neighborhood_Veenker,HouseStyle_1.5Fin,HouseStyle_1.5Unf,HouseStyle_1Story,HouseStyle_2.5Fin,HouseStyle_2.5Unf,HouseStyle_2Story,HouseStyle_SFoyer,HouseStyle_SLvl
254,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1066,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
638,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
799,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
380,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


**Combine processed continuous + categorical and train the model.**

In [9]:
X_train_processed = pd.concat([X_train_continuous, X_train_categorical], axis=1)
X_valid_processed = pd.concat([X_valid_continuous, X_valid_categorical], axis=1)

print("X_train_processed shape:", X_train_processed.shape)
print("X_valid_processed shape:", X_valid_processed.shape)
X_train_processed.head()

X_train_processed shape: (1168, 35)
X_valid_processed shape: (292, 35)


,GrLivArea,TotalBsmtSF,Neighborhood_Blmngtn,Neighborhood_Blueste,Neighborhood_BrDale,Neighborhood_BrkSide,Neighborhood_ClearCr,Neighborhood_CollgCr,Neighborhood_Crawfor,Neighborhood_Edwards,...,Neighborhood_Timber,Neighborhood_Veenker,HouseStyle_1.5Fin,HouseStyle_1.5Unf,HouseStyle_1Story,HouseStyle_2.5Fin,HouseStyle_2.5Unf,HouseStyle_2Story,HouseStyle_SFoyer,HouseStyle_SLvl
254,-0.407093,0.572612,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1066,0.083170,-0.596547,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
638,-1.395250,-0.603357,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
799,0.458975,-0.750921,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
380,0.312087,-0.081209,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [10]:
model = LinearRegression()
model.fit(X_train_processed, y_train)

joblib.dump(model, MODELS_DIR / "model.joblib")
print("Model trained and saved to", MODELS_DIR / "model.joblib")

Model trained and saved to ..\models\model.joblib


### 4.2 Model evaluation

In [11]:
y_pred_valid = model.predict(X_valid_processed)
rmsle_score = compute_rmsle(y_valid, y_pred_valid)

print("Validation RMSLE:", rmsle_score)

Validation RMSLE: 0.19


## 5. Model inference

Load all artifacts back from `../models/` (no fitting here) and predict on `test.csv`.

In [12]:
loaded_medians = joblib.load(MODELS_DIR / "continuous_medians.joblib")
loaded_modes = joblib.load(MODELS_DIR / "categorical_modes.joblib")
loaded_scaler = joblib.load(MODELS_DIR / "scaler.joblib")
loaded_encoder = joblib.load(MODELS_DIR / "encoder.joblib")
loaded_model = joblib.load(MODELS_DIR / "model.joblib")

print("All artifacts loaded from", MODELS_DIR)

All artifacts loaded from ..\models


In [13]:
test_df = pd.read_csv(TEST_PATH)
X_test = test_df[FEATURE_COLUMNS].copy()

for col in CONTINUOUS_FEATURES:
    X_test[col] = X_test[col].fillna(loaded_medians[col])
for col in CATEGORICAL_FEATURES:
    X_test[col] = X_test[col].fillna(loaded_modes[col])

X_test.head()

,GrLivArea,TotalBsmtSF,Neighborhood,HouseStyle
0,896,882.0,NAmes,1Story
1,1329,1329.0,NAmes,1Story
2,1629,928.0,Gilbert,2Story
3,1604,926.0,Gilbert,2Story
4,1280,1280.0,StoneBr,1Story


In [14]:
X_test_continuous = pd.DataFrame(
    loaded_scaler.transform(X_test[CONTINUOUS_FEATURES]),
    columns=CONTINUOUS_FEATURES,
    index=X_test.index,
)
X_test_categorical = pd.DataFrame(
    loaded_encoder.transform(X_test[CATEGORICAL_FEATURES]),
    columns=loaded_encoder.get_feature_names_out(CATEGORICAL_FEATURES),
    index=X_test.index,
)
X_test_processed = pd.concat([X_test_continuous, X_test_categorical], axis=1)
X_test_processed.head()

,GrLivArea,TotalBsmtSF,Neighborhood_Blmngtn,Neighborhood_Blueste,Neighborhood_BrDale,Neighborhood_BrkSide,Neighborhood_ClearCr,Neighborhood_CollgCr,Neighborhood_Crawfor,Neighborhood_Edwards,...,Neighborhood_Timber,Neighborhood_Veenker,HouseStyle_1.5Fin,HouseStyle_1.5Unf,HouseStyle_1Story,HouseStyle_2.5Fin,HouseStyle_2.5Unf,HouseStyle_2Story,HouseStyle_SFoyer,HouseStyle_SLvl
0,-1.204486,-0.408119,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,-0.378479,0.606665,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,0.193813,-0.303689,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,0.146122,-0.308230,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,-0.471953,0.495425,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0


In [15]:
predictions = loaded_model.predict(X_test_processed)
print("Predictions shape:", predictions.shape)
print("First 5 predictions:", predictions[:5])

Predictions shape: (1459,)
First 5 predictions: [112528.98609809 157159.79069386 191541.74712742 189564.10291849
 258664.0794165 ]
